# Tree Matcher Demo Notebook

This notebook gives sample syntax for the following steps:

1. **Sampling** labelled trees with a **planted root→leaf path** (toy model from the paper).
2. **Fast matching for the common simple weight families**. This is the default for most single matches.
3. **Pre-encoded fast matching for batch processing**. This is the default when you will match many pairs from one dataset.
4. **Generic matching with a custom weight function**. This is slower, but more flexible.
5. **Plotting / sanity checks**.

The key practical point is:

- Use `FastTreePathMatcher(...)` first when your weight is either
  - **equality only**: identical labels score a token weight, otherwise `0`, or
  - **overlap / any-common-token** on set-like labels.
- Use the original `TreePathMatcher(...)` matchers when you need a more general custom `w(label_u, label_v)`.


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "path_matcher").exists():
            return candidate
    raise RuntimeError("Could not find repository root containing pyproject.toml and src/path_matcher.")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
for path in (str(REPO_ROOT), str(SRC)):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Repo root:", REPO_ROOT)


In [ ]:
# Imports + repo path

import os, sys
import numpy as np
import matplotlib.pyplot as plt
from path_matcher.matcher import TreePathMatcher
from path_matcher.fast_match import FastTreePathMatcher, HAVE_NUMBA

In [ ]:


# If this notebook sits in e.g. <repo>/notebooks/, uncomment and adjust:
# REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), ".."))
# sys.path.insert(0, REPO_ROOT)

# Local modules (adjust if your layout differs)
from path_matcher.planted_path_sampler import (
    ToyModelConfig,
    GaltonWatsonTreeSampler,
    make_default_tree_samplers,
    sample_toy_model,
)

from path_matcher.tree_viz import (
    PlotStyle,
    plot_tree_with_path,
    build_discrete_label_colormap,
    vertex_colors_by_label,
    vertex_labels_from_ordered_vertices,
)

from path_matcher.matcher import TreePathMatcher

# Fast matcher: place fast_match.py inside the path_matcher package.
# The fallback import below also lets this notebook run in a loose-file setup.
try:
    from path_matcher.fast_match import FastTreePathMatcher
except Exception:
    from fast_match import FastTreePathMatcher

from path_matcher.demo import *


## 1. Sampler demo

We’ll sample a small toy dataset. This is where you’ll typically vary:
- Galton–Watson parameters: `lam`, `max_depth`, `max_nodes`, and the rejection constraints.
- Planting/label parameters: `alphabet_size`, `planted_seq_len`, and `p_obs` (fraction of path positions overwritten by the planted sequence).
- Number of classes and graphs: `n_classes`, `n_per_class`.

We’ll also show the two default GW variants:
- BFS expansion (`traversal="bfs"`)
- DFS expansion (`traversal="dfs"`)

(These differ only because we prune at `max_nodes`.)

In [ ]:
# Sampling configuration (tweak freely)

seed = 0
rng = np.random.default_rng(seed)

# Two default samplers: BFS vs DFS expansion order
samplers = make_default_tree_samplers(
    max_depth=10,
    lam=1.7,
    max_nodes=160,        # prune at this size
    min_nodes=120,        # reject if smaller
    require_full_depth=True,
    max_tries=5000,
)

cfg = ToyModelConfig(
    n_classes=3,
    n_per_class=4,
    planted_seq_len=12,
    alphabet_size=15,
    p_obs=1.0,
    dirichlet_alpha=1.0,
    seed=seed,
)

# Sample with BFS variant (swap to samplers["gw_dfs"] to compare)
graphs, class_ids, class_sequences = sample_toy_model(cfg, tree_path_sampler=samplers["gw_bfs"])

print(f"N graphs: {len(graphs)}")
print(f"Class label counts: {np.bincount(np.asarray(class_ids, dtype=int))}")
print("Example planted sequences (per class):")
for i, seq in enumerate(class_sequences):
    print(f"  class {i}: {seq}")

# Add a depth feature for plotting
for G in graphs:
    annotate_depth_as_feature(G, root=0, attr="dp_features")

### Visual sanity-check: a few sampled trees (with ground truth planted path)

This uses:
- vertex colors = `dp_features` (depth),
- blue highlight = planted path (from `is_planted`).

In [ ]:
# Plot a few sampled trees with their planted paths (ground truth)

style_truth = PlotStyle(
    path_edge_color="dodgerblue",
    path_vertex_frame_color="dodgerblue",
)

k_show = min(3, len(graphs))
for i in range(k_show):
    G = graphs[i]
    planted_path = extract_planted_path_ordered(G)

    # Background = black; highlighted (truth) vertices colored by their categorical label
    colors, _ = vertex_colors_by_label(
        G,
        "label",
        highlight_vertices=planted_path,
        background_color="black",
        colormap="viridis",
    )

    fig, ax = plot_tree_with_path(
        G,
        path_vertices=planted_path,
        vertex_color=colors,
        color_by_attr=None,
        style=style_truth,
        figsize=(6, 6),
        show_vertex_labels=False,
    )
    ax.set_title(f"Sample tree {i} (class={class_ids[i]}) — planted path (truth)")
    plt.show()


## 2. Fast matcher demo (default)

For the two most common scoring families, the **default** matcher is `FastTreePathMatcher`.

It supports two special cases:

1. **Equality scoring**
   \[
   w(a,b) = \begin{cases}
   \text{token\_weight}(a), & a=b,\\
   0, & a\neq b.
   \end{cases}
   \]

2. **Overlap / any-common-token scoring** for set-like or multiset-like labels
   \[
   w(A,B) = \max\{\text{token\_weight}(t) : t \in A \cap B\},
   \]
   with the unweighted version being simply “any overlap gives 1, otherwise 0”.

These cover the usual single-token case, and many multi-label cases as well.  
The current toy sampler uses **scalar labels**, so below we demonstrate `mode="equality"`.

For multi-label data, the typical pattern is something like

```python
FastTreePathMatcher(mode="overlap", label_getter=lambda label: label["Labels"])
```

### Default workflows

- **Single match:** `fit(G, H)` then `predict()`.
- **Batch processing:** `fit_encoder(graphs)` once, pre-encode each tree once, then call `predict_encoded(...)`.


In [ ]:
# Choose a pair to match: by default, same class if possible

def pick_pair_same_class(class_ids, class_id=0):
    idx = [i for i, c in enumerate(class_ids) if c == class_id]
    if len(idx) < 2:
        return 0, 1
    return idx[0], idx[1]

i, j = pick_pair_same_class(class_ids, class_id=0)
G = graphs[i]
H = graphs[j]

print(f"Matching G=graphs[{i}] (class {class_ids[i]}) vs H=graphs[{j}] (class {class_ids[j]})")

# Ground truth planted paths (for comparison)
truth_G = extract_planted_path_ordered(G)
truth_H = extract_planted_path_ordered(H)
print(f"|truth_G|={len(truth_G)}, |truth_H|={len(truth_H)}")

# Fast-friendly equality weight used in Section 2
def w_fast_eq(a, b):
    return 1.0 if a == b else 0.0


### 2a) Fast single-match API (`FastTreePathMatcher(mode="equality")`)

For one-off comparisons, the recommended API is:

1. construct `FastTreePathMatcher(...)`,
2. call `fit(G, H)`,
3. call `predict()`.

Here we use the simple equality score `1` for an exact label match and `0` otherwise.


In [ ]:
# Fast matcher: single pair (default for simple equality / overlap-style weights)

fast_single = FastTreePathMatcher(
    mode="equality",
    default_weight=1.0,
)

fast_single.fit(G, H)
pairs_fast, score_fast = fast_single.predict()

# Optional sanity check on one pair: compare against the generic exact matcher
matcher_exact_fast_ref = TreePathMatcher(
    method="exact",
    w=w_fast_eq,
)
matcher_exact_fast_ref.fit(G, H)
pairs_fast_ref, score_fast_ref = matcher_exact_fast_ref.predict()

matchG_fast = [int(a) for (a, _b) in pairs_fast]
matchH_fast = [int(b) for (_a, b) in pairs_fast]

print(f"Fast score: {score_fast}")
print(f"Generic exact score for the same simple weight: {score_fast_ref}")
print(f"Scores agree: {np.isclose(score_fast, score_fast_ref)}")
print(f"Number of matched pairs: {len(pairs_fast)}")
print("First few matched pairs:", pairs_fast[:10])


In [ ]:
# Plot estimated matches vs truth (Fast equality matcher)

score_truth_fast = score_fixed_paths_via_matcher(G, truth_G, H, truth_H, w_fast_eq, method="exact")

style_overlay = PlotStyle(
    path_edge_color="red",
    path_vertex_frame_color="red",
    path2_edge_color="dodgerblue",
    path2_vertex_frame_color="dodgerblue",
)

matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G,
    H,
    pairs_fast,
    truth_G=truth_G,
    truth_H=truth_H,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

plot_tree_with_path(
    G,
    path_vertices=matchG_viz,
    path2_vertices=truth_G,
    ax=axes[0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0].set_title(
    f"Tree G: fast est (red) vs truth (blue) — est={score_fast:.3g}, truth={score_truth_fast:.3g}"
)

plot_tree_with_path(
    H,
    path_vertices=matchH_viz,
    path2_vertices=truth_H,
    ax=axes[1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1].set_title(
    f"Tree H: fast est (red) vs truth (blue) — est={score_fast:.3g}, truth={score_truth_fast:.3g}"
)

plt.show()


### 2b) Fast batch API (`fit_encoder(...)` + `predict_encoded(...)`)

When you will match **many pairs from one dataset**, the default workflow is to pre-encode once:

1. fit one encoder on the dataset,
2. encode each tree once,
3. call `predict_encoded(...)` for each pair.

This avoids repeated label normalization / integerization work.


In [ ]:
# Fast matcher: pre-encoded batch workflow (default for many pairwise matches)

fast_batch = FastTreePathMatcher(
    mode="equality",
    default_weight=1.0,
)

# Fit one shared encoder, then pre-encode each tree once
fast_batch.fit_encoder(graphs)
graphs_fast = [fast_batch.encode_tree(T) for T in graphs]

pairs_fast_batch, score_fast_batch = fast_batch.predict_encoded(graphs_fast[i], graphs_fast[j])

print(f"Pre-encoded score for the same pair: {score_fast_batch}")
print(f"Matches single-pair fast score: {np.isclose(score_fast_batch, score_fast)}")
print("First few matched pairs:", pairs_fast_batch[:10])

# Small example score matrix on a few graphs
k_batch = min(5, len(graphs_fast))
score_mat = np.full((k_batch, k_batch), np.nan)

for a in range(k_batch):
    for b in range(a + 1, k_batch):
        _pairs_ab, s_ab = fast_batch.predict_encoded(graphs_fast[a], graphs_fast[b])
        score_mat[a, b] = score_mat[b, a] = s_ab

print("Small pre-encoded score matrix (first few graphs):")
print(np.array_str(score_mat, precision=2, suppress_small=True))


## 3. Generic matcher demo (slower, for custom weight functions)

Use the original `TreePathMatcher(...)` methods below when you need a **custom**
`w(label_u, label_v)` that does **not** fit the fast equality / overlap families above.

That is the more flexible route, but it is slower.

In the examples below, we use a mismatch penalty,
```python
w(a, b) = 1.0 if a == b else -0.25
```
which is outside the fast special case, so it belongs in this generic section.


### 3a) Exact matcher (`method="exact"`)

This calls the full dynamic program for a completely general custom weight function `w(label_u, label_v)`.

We’ll plot:
- estimated matched vertices (red),
- planted-path truth (blue),
- and titles showing both the estimated score and the planted-path score.


In [ ]:
# Exact matcher with a custom weight function

def w(a, b):
    return 1.0 if a == b else -0.25

matcher_exact = TreePathMatcher(
    method="exact",
    w=w,
)

matcher_exact.fit(G, H)
pairs_exact, score_exact = matcher_exact.predict()

# IMPORTANT: the matcher returns a *sequence of matched vertex pairs*.
# These matched vertices are typically NOT a connected path in either tree.
matchG_exact = [int(a) for (a, _b) in pairs_exact]
matchH_exact = [int(b) for (_a, b) in pairs_exact]

print(f"Exact score: {score_exact}")
print(f"Number of matched pairs: {len(pairs_exact)}")
print("First few matched pairs:", pairs_exact[:10])


In [ ]:
# Plot estimated matches vs truth (Exact)

score_truth_exact = score_fixed_paths_via_matcher(G, truth_G, H, truth_H, w, method="exact")

style_overlay = PlotStyle(
    path_edge_color="red",
    path_vertex_frame_color="red",
    path2_edge_color="dodgerblue",
    path2_vertex_frame_color="dodgerblue",
)

matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G,
    H,
    pairs_exact,
    truth_G=truth_G,
    truth_H=truth_H,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

plot_tree_with_path(
    G,
    path_vertices=matchG_viz,   # estimated matched vertices (not necessarily connected)
    path2_vertices=truth_G,     # truth
    ax=axes[0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0].set_title(
    f"Tree G: est (red) vs truth (blue) — est={score_exact:.3g}, truth={score_truth_exact:.3g}"
)

plot_tree_with_path(
    H,
    path_vertices=matchH_viz,
    path2_vertices=truth_H,
    ax=axes[1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1].set_title(
    f"Tree H: est (red) vs truth (blue) — est={score_exact:.3g}, truth={score_truth_exact:.3g}"
)

plt.show()


### 3b) Beam matcher (`method="beam"`)

This runs the pruned / beam variant for the same custom weight `w`.

Useful parameters to play with:
- `beam_width` (larger = less pruning, more work),
- `match_predicate` (quick admissibility test for candidate label pairs).


In [ ]:
# Beam matcher

matcher_beam = TreePathMatcher(
    method="beam",
    w=w,
    beam_width=200,                 # tune
    match_predicate=lambda a, b: (a == b),
)

matcher_beam.fit(G, H)
pairs_beam, score_beam = matcher_beam.predict()

matchG_beam = [int(a) for (a, _b) in pairs_beam]
matchH_beam = [int(b) for (_a, b) in pairs_beam]

print(f"Beam score: {score_beam}")
print(f"Number of matched pairs: {len(pairs_beam)}")
print("First few matched pairs:", pairs_beam[:10])


In [ ]:
# Plot estimated matches vs truth (Beam)

score_truth_beam = score_fixed_paths_via_matcher(G, truth_G, H, truth_H, w, method="exact")

matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G,
    H,
    pairs_beam,
    truth_G=truth_G,
    truth_H=truth_H,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

plot_tree_with_path(
    G,
    path_vertices=matchG_viz,
    path2_vertices=truth_G,
    ax=axes[0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0].set_title(
    f"Tree G: beam est (red) vs truth (blue) — est={score_beam:.3g}, truth={score_truth_beam:.3g}"
)

plot_tree_with_path(
    H,
    path_vertices=matchH_viz,
    path2_vertices=truth_H,
    ax=axes[1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1].set_title(
    f"Tree H: beam est (red) vs truth (blue) — est={score_beam:.3g}, truth={score_truth_beam:.3g}"
)

plt.show()


### 3c) Bucketed / sparse matcher (`method="sparse"`) using a bucketable weight

This is still part of the **generic** workflow: it is for a user-defined weight function `w`,
but one that is structured enough to support bucketing / blocking.

Below we deliberately start with a plain `w` that has **no** such method, then wrap it
using `make_bucketable_weight(...)`.


In [ ]:
# Sparse / bucketed matcher (illustrates wrapping a non-bucketable w)

def w_plain(a, b):
    return 1.0 if a == b else -0.25

print("w_plain has blocking_keys:", hasattr(w_plain, "blocking_keys"))

from path_matcher.weight_wrappers import make_bucketable_weight

# Wrap w_plain to add blocking keys / bucketing support
w_bucket = make_bucketable_weight(w_plain, scheme="token_overlap")

print("w_bucket has blocking_keys:", hasattr(w_bucket, "blocking_keys"))

matcher_sparse = TreePathMatcher(
    method="sparse",
    w=w_bucket,
)

matcher_sparse.fit(G, H)
pairs_sparse, score_sparse = matcher_sparse.predict()

matchG_sparse = [int(a) for (a, _b) in pairs_sparse]
matchH_sparse = [int(b) for (_a, b) in pairs_sparse]

print(f"Sparse/bucketed score: {score_sparse}")
print(f"Number of matched pairs: {len(pairs_sparse)}")
print("First few matched pairs:", pairs_sparse[:10])


In [ ]:
# Plot estimated matches vs truth (Sparse/bucketed)

score_truth_sparse = score_fixed_paths_via_matcher(G, truth_G, H, truth_H, w_bucket, method="exact")

matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G,
    H,
    pairs_sparse,
    truth_G=truth_G,
    truth_H=truth_H,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

plot_tree_with_path(
    G,
    path_vertices=matchG_viz,
    path2_vertices=truth_G,
    ax=axes[0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0].set_title(
    f"Tree G: sparse est (red) vs truth (blue) — est={score_sparse:.3g}, truth={score_truth_sparse:.3g}"
)

plot_tree_with_path(
    H,
    path_vertices=matchH_viz,
    path2_vertices=truth_H,
    ax=axes[1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1].set_title(
    f"Tree H: sparse est (red) vs truth (blue) — est={score_sparse:.3g}, truth={score_truth_sparse:.3g}"
)

plt.show()


## 4. Just plotting.

Sometimes it’s helpful to see just the planted path without any estimates.


In [ ]:
style_truth_only = PlotStyle(
    path_edge_color="dodgerblue",
    path_vertex_frame_color="dodgerblue",
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

colorsG_truth, _ = vertex_colors_by_label(
    G,
    "label",
    highlight_vertices=truth_G,
    background_color="black",
    colormap="viridis",
)
plot_tree_with_path(
    G,
    path_vertices=truth_G,
    ax=axes[0],
    vertex_color=colorsG_truth,
    color_by_attr=None,
    style=style_truth_only,
    show_vertex_labels=False,
)
axes[0].set_title("Tree G: planted path (truth)")

colorsH_truth, _ = vertex_colors_by_label(
    H,
    "label",
    highlight_vertices=truth_H,
    background_color="black",
    colormap="viridis",
)
plot_tree_with_path(
    H,
    path_vertices=truth_H,
    ax=axes[1],
    vertex_color=colorsH_truth,
    color_by_attr=None,
    style=style_truth_only,
    show_vertex_labels=False,
)
axes[1].set_title("Tree H: planted path (truth)")

plt.show()


## 5. Optional: compare same-class vs different-class matches

The goal here is to sanity-check that the matching score (and the highlighted path) is
**better when the planted sequences agree** (same class) than when they differ (different classes).


In [ ]:
# Compare same-class vs different-class matches (Exact)

# Same-class pair
i_same, j_same = pick_pair_same_class(class_ids, class_id=0)
G_same = graphs[i_same]
H_same = graphs[j_same]
truth_G_same = extract_planted_path_ordered(G_same)
truth_H_same = extract_planted_path_ordered(H_same)

m_same = TreePathMatcher(method="exact", w=w)
m_same.fit(G_same, H_same)
pairs_same, score_same = m_same.predict()

matchG_same = [int(a) for (a, _b) in pairs_same]
matchH_same = [int(b) for (_a, b) in pairs_same]

score_truth_same = score_fixed_paths_via_matcher(G_same, truth_G_same, H_same, truth_H_same, w, method="exact")

# Different-class pair
def pick_pair_diff_class(class_ids):
    for a in range(len(class_ids)):
        for b in range(a + 1, len(class_ids)):
            if class_ids[a] != class_ids[b]:
                return a, b
    return 0, 1

i_diff, j_diff = pick_pair_diff_class(class_ids)
G_diff = graphs[i_diff]
H_diff = graphs[j_diff]
truth_G_diff = extract_planted_path_ordered(G_diff)
truth_H_diff = extract_planted_path_ordered(H_diff)

m_diff = TreePathMatcher(method="exact", w=w)
m_diff.fit(G_diff, H_diff)
pairs_diff, score_diff = m_diff.predict()

matchG_diff = [int(a) for (a, _b) in pairs_diff]
matchH_diff = [int(b) for (_a, b) in pairs_diff]

score_truth_diff = score_fixed_paths_via_matcher(G_diff, truth_G_diff, H_diff, truth_H_diff, w, method="exact")

print(f"Same class: i={i_same} (class {class_ids[i_same]}) vs j={j_same} (class {class_ids[j_same]})  -> est={score_same}, truth={score_truth_same}")
print(f"Diff class: i={i_diff} (class {class_ids[i_diff]}) vs j={j_diff} (class {class_ids[j_diff]})  -> est={score_diff}, truth={score_truth_diff}")

fig, axes = plt.subplots(2, 2, figsize=(12, 12))

# Row 0: same-class
matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G_same,
    H_same,
    pairs_same,
    truth_G=truth_G_same,
    truth_H=truth_H_same,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

plot_tree_with_path(
    G_same,
    path_vertices=matchG_viz,
    path2_vertices=truth_G_same,
    ax=axes[0, 0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0, 0].set_title(f"Same class — Tree G — est={score_same:.3g}, truth={score_truth_same:.3g}")

plot_tree_with_path(
    H_same,
    path_vertices=matchH_viz,
    path2_vertices=truth_H_same,
    ax=axes[0, 1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[0, 1].set_title(f"Same class — Tree H — est={score_same:.3g}, truth={score_truth_same:.3g}")

# Row 1: different-class
matchG_viz, matchH_viz, colorsG, colorsH, labelsG, labelsH = prep_match_plot_inputs(
    G_diff,
    H_diff,
    pairs_diff,
    truth_G=truth_G_diff,
    truth_H=truth_H_diff,
    label_attr="label",
    colormap="viridis",
    background_color="black",
)

plot_tree_with_path(
    G_diff,
    path_vertices=matchG_viz,
    path2_vertices=truth_G_diff,
    ax=axes[1, 0],
    vertex_color=colorsG,
    color_by_attr=None,
    vertex_label=labelsG,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1, 0].set_title(f"Diff class — Tree G — est={score_diff:.3g}, truth={score_truth_diff:.3g}")

plot_tree_with_path(
    H_diff,
    path_vertices=matchH_viz,
    path2_vertices=truth_H_diff,
    ax=axes[1, 1],
    vertex_color=colorsH,
    color_by_attr=None,
    vertex_label=labelsH,
    style=style_overlay,
    show_vertex_labels=False,
)
axes[1, 1].set_title(f"Diff class — Tree H — est={score_diff:.3g}, truth={score_truth_diff:.3g}")

plt.tight_layout()
plt.show()
